# Multi-Task NLP Intelligence Suite — Phase 1
## Exploring HuggingFace Pipelines

**Project:** Multi-Task NLP Intelligence Suite  
**Phase:** 1 of 5 — Pipeline Exploration  
**Author:** Amiru AI Lab

---

### What we cover in this notebook

1.  Summarization `facebook/bart-large-cnn` Compress long text

2.  Sentiment Analysis  `cardiffnlp/twitter-roberta-base-sentiment-latest` Positive / Negative / Neutral

3.  Zero-Shot Classification  `facebook/bart-large-mnli`  Classify without training

4.  Named Entity Recognition  `dslim/bert-base-NER`  Extract people, orgs, locations

5.  Question Answering `deepset/roberta-base-squad2` Extract answers from context

> **Phase 1 goal:** Run each pipeline, understand its output schema, handle edge cases, and log confidence scores. This forms the foundation for the FastAPI backend in Phase 2.

---
## 🔧 Setup & Installation

In [66]:
# Run once — installs all required packages

!pip install transformers==4.51.3 torch sentencepiece accelerate -q
print(" Installation complete")

 Installation complete


In [67]:
from transformers import pipeline
import json
import time
import warnings
warnings.filterwarnings('ignore')

# ── Utility: pretty-print any pipeline result ──────────────────────────────
def show(label, result):
    """Prints a labelled, formatted JSON result."""
    print(f"\n{'─'*55}")
    print(f" {label}")
    print(f"{'─'*55}")
    print(json.dumps(result, indent=2, ensure_ascii=False))

# ── Utility: measure inference time ────────────────────────────────────────
def timed_run(pipe_fn, *args, **kwargs):
    """Runs a pipeline call and returns (result, elapsed_ms)."""
    start = time.time()
    result = pipe_fn(*args, **kwargs)
    elapsed = round((time.time() - start) * 1000)
    return result, elapsed

print("Utilities loaded")

Utilities loaded


---
## 📝 Shared Test Corpus

We define all test texts once here. Each pipeline will reuse these, which lets us **compare how different models interpret the same input** — a key skill for model selection in production.

In [68]:
# ── Long-form article for summarization ────────────────────────────────────
LONG_TEXT = """
Artificial intelligence is transforming how companies build and deploy software.
Large language models like GPT-4 and Claude can now generate production-quality code,
write technical documentation, and debug complex issues in seconds. Several major
tech companies have reported 30-50% productivity gains among their engineering teams
after adopting AI-assisted coding tools. However, concerns remain around code quality,
security vulnerabilities introduced by auto-generated code, and the long-term impact
on junior developer roles. Critics argue that over-reliance on AI tools may erode
fundamental programming skills among new engineers. Proponents counter that these
tools free developers from repetitive boilerplate and allow them to focus on
higher-level system design and architecture. The debate continues as adoption accelerates.
""".strip()

# ── Short texts for sentiment & NER ───────────────────────────────────────
TEXTS = [
    "I absolutely love how HuggingFace makes state-of-the-art NLP so accessible!",
    "The model was painfully slow and the documentation is completely outdated.",
    "The library released version 4.0 last Tuesday.",  # Neutral
    "Elon Musk announced that Tesla will open a new factory in Berlin next year.",
    "Apple and Google are collaborating on a new AI safety initiative in California.",
]

# ── Context + questions for QA ─────────────────────────────────────────────
QA_CONTEXT = """
HuggingFace was founded in 2016 by Clement Delangue, Julien Chaumond, and Thomas Wolf.
The company is headquartered in New York City and Paris. It is best known for its
Transformers library which provides thousands of pre-trained models for NLP, computer
vision, and audio tasks. The Hub hosts over 500,000 models and 100,000 datasets.
In 2023, HuggingFace raised a Series D funding round valuing the company at $4.5 billion.
""".strip()

QA_QUESTIONS = [
    "Who founded HuggingFace?",
    "Where is HuggingFace headquartered?",
    "How many models does the Hub host?",
    "What was HuggingFace valued at in 2023?",
]

# ── Zero-shot topics ────────────────────────────────────────────────────────
ZS_TEXTS = [
    "The new transformer architecture reduces inference time by 40% on benchmark tasks.",
    "The president signed a new climate bill into law amid rising global temperatures.",
    "Manchester United won 3-1 against Chelsea in last night's Premier League match.",
]
ZS_LABELS = ["technology", "politics", "sports", "business", "science", "entertainment"]

print("Test corpus ready")
print(f"   Long text  : {len(LONG_TEXT.split())} words")
print(f"   Short texts: {len(TEXTS)} samples")
print(f"   QA context : {len(QA_CONTEXT.split())} words, {len(QA_QUESTIONS)} questions")
print(f"   ZS texts   : {len(ZS_TEXTS)} samples, {len(ZS_LABELS)} candidate labels")

Test corpus ready
   Long text  : 111 words
   Short texts: 5 samples
   QA context : 68 words, 4 questions
   ZS texts   : 3 samples, 6 candidate labels


---
## Pipeline 1 — Summarization
**Model:** `facebook/bart-large-cnn`  
**What it does:** Abstractive summarization — generates new sentences, doesn't just copy sentences from the source.

**Key parameters to understand:**
- `max_length` / `min_length` — control summary length in tokens
- `do_sample=False` — use beam search (deterministic), not sampling
- `num_beams` — how many beams in beam search (higher = better quality, slower)

> **Token limit:** BART accepts up to **1024 tokens**. Text longer than this needs chunking — we handle that in the edge case cell below.

In [69]:
print("Loading summarization model... (downloads ~1.6GB on first run)")
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0  # Uncomment if you have a GPU
)
print("Summarizer ready")

Loading summarization model... (downloads ~1.6GB on first run)


Device set to use cpu


Summarizer ready


In [70]:
# ── Basic summarization ─────────────────────────────────────────────────────
result, ms = timed_run(
    summarizer,
    LONG_TEXT,
    max_length=80,
    min_length=25,
    do_sample=False
)

show(f"Summarization ({ms}ms)", result)
summary_text = result[0]['summary_text']
print(f"\n Compression ratio: {len(LONG_TEXT.split())} words → {len(summary_text.split())} words")


───────────────────────────────────────────────────────
 Summarization (20817ms)
───────────────────────────────────────────────────────
[
  {
    "summary_text": "Large language models like GPT-4 and Claude can now generate production-quality code. Critics argue that over-reliance on AI tools may erode fundamental programming skills. Proponents counter that these tools free developers from repetitive boilerplate."
  }
]

 Compression ratio: 111 words → 34 words


In [71]:
# ── Experiment: how do different max_length values change the output? ────────
print("Comparing summary lengths:\n")
for max_len in [30, 60, 120]:
    r = summarizer(LONG_TEXT, max_length=max_len, min_length=10, do_sample=False)
    txt = r[0]['summary_text']

    print(f"max_length={max_len:>3} - [{len(txt.split()):>2} words] {txt[:120]}...")

Comparing summary lengths:

max_length= 30 - [18 words] Large language models like GPT-4 and Claude can now generate production-quality code. Critics argue that over-reliance o...
max_length= 60 - [34 words] Large language models like GPT-4 and Claude can now generate production-quality code. Critics argue that over-reliance o...
max_length=120 - [34 words] Large language models like GPT-4 and Claude can now generate production-quality code. Critics argue that over-reliance o...


In [14]:
# ── Edge case: text that exceeds 1024 tokens ─────────────────────────────────

def chunk_and_summarize(text, pipe, chunk_size=900, max_length=80, min_length=20):
    """
    Splits text into word-based chunks, summarizes each, then
    optionally summarizes the combined summaries (map-reduce style).
    """
    words = text.split()
    chunks = [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

    print(f" Split into {len(chunks)} chunk(s)")

    summaries = []
    for i, chunk in enumerate(chunks):
        r = pipe(chunk, max_length=max_length, min_length=min_length, do_sample=False)
        summaries.append(r[0]['summary_text'])
        print(f"  Chunk {i+1}: {r[0]['summary_text'][:80]}...")

    # If multiple chunks, do a second-pass summary
    if len(summaries) > 1:
        combined = ' '.join(summaries)
        final = pipe(combined, max_length=max_length, min_length=min_length, do_sample=False)
        return final[0]['summary_text']
    return summaries[0]

# Simulate a long document (repeat our text)
very_long = (LONG_TEXT + ' ') * 5
print(f"Long doc: {len(very_long.split())} words\n")

final_summary = chunk_and_summarize(very_long, summarizer)
print(f"\n Final summary: {final_summary}")

Long doc: 555 words

 Split into 1 chunk(s)
  Chunk 1: Artificial intelligence is transforming how companies build and deploy software....

 Final summary: Artificial intelligence is transforming how companies build and deploy software. Large language models like GPT-4 and Claude can now generate production-quality code, write technical documentation, and debug complex issues in seconds. Critics argue that over-reliance on AI tools may erode fundamental programming skills among new engineers. Proponents counter that these tools free developers from repetitive boilerplate and allow them to


---
## Pipeline 2 — Sentiment Analysis
**Model:** `cardiffnlp/twitter-roberta-base-sentiment-latest`  
**What it does:** Classifies text as `positive`, `negative`, or `neutral` with a confidence score.

**Why this model vs `distilbert-base-uncased-finetuned-sst-2-english`?**  
The Cardiff model was trained on 124M tweets so it handles informal language, slang, and social media text better. The distilbert version is better for formal text. We'll compare both below.

In [13]:
print("Loading sentiment models...")
sentiment_primary = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    return_all_scores=True  # Get scores for ALL labels, not just the top one
)

sentiment_alt = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    return_all_scores=True
)
print("Both sentiment models ready")

Loading sentiment models...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


...

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


Both sentiment models ready


In [17]:
# ── Run all 5 test texts through the primary model ──────────────────────────
print("=" * 60)
print(" Cardiff RoBERTa — Sentiment Results")
print("=" * 60)

sentiment_log = []  # We'll use this for the confidence score analysis

for text in TEXTS:
    result, ms = timed_run(sentiment_primary, text)
    scores = result[0]  # List of {label, score} dicts
    top = max(scores, key=lambda x: x['score'])

    # Build a visual confidence bar
    bar_len = int(top['score'] * 20)
    bar = "#" * bar_len + "!" * (20 - bar_len)

    print(f"\nText: {text[:70]}..." if len(text) > 70 else f"\nText: {text}")
    print(f"  → {top['label'].upper():<10} [{bar}] {top['score']:.1%}  ({ms}ms)")

    sentiment_log.append({'text': text, 'label': top['label'], 'score': top['score'], 'all': scores})

 Cardiff RoBERTa — Sentiment Results

Text: I absolutely love how HuggingFace makes state-of-the-art NLP so access...
  → POSITIVE   [###################!] 97.9%  (133ms)

Text: The model was painfully slow and the documentation is completely outda...
  → NEGATIVE   [##################!!] 95.0%  (118ms)

Text: The library released version 4.0 last Tuesday.
  → NEUTRAL    [################!!!!] 84.1%  (94ms)

Text: Elon Musk announced that Tesla will open a new factory in Berlin next ...
  → POSITIVE   [###########!!!!!!!!!] 55.8%  (118ms)

Text: Apple and Google are collaborating on a new AI safety initiative in Ca...
  → NEUTRAL    [#############!!!!!!!] 66.5%  (107ms)


In [22]:
# ── Model comparison: Cardiff vs DistilBERT on the same texts ─────────────
# KEY LEARNING: Different models can disagree — especially on neutral text!

print(f"{'Text':<45} {'Cardiff':<18} {'DistilBERT':<18}")
print("─" * 82)

def normalize_label(label):
    label = label.upper()

    if label == "LABEL_0":
        return "NEGATIVE"
    elif label == "LABEL_1":
        return "POSITIVE"

    return label

for text in TEXTS:
    r1 = sentiment_primary(text)[0]
    r2 = sentiment_alt(text)[0]

    top1 = max(r1, key=lambda x: x['score'])
    top2 = max(r2, key=lambda x: x['score'])

    l1 = normalize_label(top1['label'])
    l2 = normalize_label(top2['label'])

    short_text = (text[:42] + '...') if len(text) > 42 else text.ljust(45)
    c1 = f"{l1} {top1['score']:.0%}"
    c2 = f"{l2} {top2['score']:.0%}"

    flag = " DISAGREE" if l1 != l2 else ""
    print(f"{short_text:<45} {c1:<18} {c2:<18}{flag}")


print("\n Takeaway: DistilBERT has no 'neutral' class — it forces everything into positive/negative.")
print("   Cardiff RoBERTa handles neutral text correctly. Choose based on your use case!")

Text                                          Cardiff            DistilBERT        
──────────────────────────────────────────────────────────────────────────────────
I absolutely love how HuggingFace makes st... POSITIVE 98%       POSITIVE 100%     
The model was painfully slow and the docum... NEGATIVE 95%       NEGATIVE 100%     
The library released version 4.0 last Tues... NEUTRAL 84%        NEGATIVE 86%       DISAGREE
Elon Musk announced that Tesla will open a... POSITIVE 56%       POSITIVE 100%     
Apple and Google are collaborating on a ne... NEUTRAL 66%        POSITIVE 64%       DISAGREE

 Takeaway: DistilBERT has no 'neutral' class — it forces everything into positive/negative.
   Cardiff RoBERTa handles neutral text correctly. Choose based on your use case!


In [24]:
# ── Edge case: confidence thresholding ──────────────────────────────────────
# In production, low-confidence predictions should be flagged, not trusted blindly

CONFIDENCE_THRESHOLD = 0.65  # Below this = "uncertain"

edge_texts = [
    "It was okay, I guess.",             # Ambiguous
    "The product arrived.",              # Purely factual
    "Not bad, could be worse.",          # Double negative
    "This is the worst best thing ever.", # Contradictory
]

print("Confidence thresholding edge cases:\n")
for text in edge_texts:
    r = sentiment_primary(text)[0]
    top = max(r, key=lambda x: x['score'])
    flag = ":" if top['score'] >= CONFIDENCE_THRESHOLD else ": LOW CONFIDENCE"

    print(f"  {flag} '{text}'")
    print(f"       → {top['label'].upper()} ({top['score']:.1%})\n")

print(" Sentiment endpoint will return a 'low_confidence' flag")
print(f"  when score < {CONFIDENCE_THRESHOLD}.")

Confidence thresholding edge cases:

  : 'It was okay, I guess.'
       → POSITIVE (72.1%)

  : 'The product arrived.'
       → POSITIVE (68.9%)

  : LOW CONFIDENCE 'Not bad, could be worse.'
       → NEUTRAL (45.9%)

  : 'This is the worst best thing ever.'
       → NEGATIVE (77.4%)

 Sentiment endpoint will return a 'low_confidence' flag
  when score < 0.65.


---
##  Pipeline 3 — Zero-Shot Classification
**Model:** `facebook/bart-large-mnli`  

**This is the most conceptually interesting pipeline.** MNLI = Multi-Genre Natural Language Inference. The model was trained to determine if a "hypothesis" follows from a "premise". Zero-shot classification repurposes this: it checks whether `"This text is about {label}"` is entailed by your input text.


In [25]:
print("Loading zero-shot classification model...")
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
print("Zero-shot classifier ready")

Loading zero-shot classification model...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Zero-shot classifier ready


In [27]:
# ── Run all test texts with our candidate labels ─────────────────────────────
print("Candidate labels:", ZS_LABELS)
print("\n" + "=" * 60)

for text in ZS_TEXTS:
    result, ms = timed_run(
        zero_shot, text,
        candidate_labels=ZS_LABELS,
        multi_label=False  # Single most likely label
    )
    print(f"\n Text: {text}")
    print(f"   Top label : {result['labels'][0].upper()} ({result['scores'][0]:.1%})  [{ms}ms]")
    print(f"   Runner-up : {result['labels'][1].upper()} ({result['scores'][1]:.1%})")

    # Bar chart for all label scores
    for label, score in zip(result['labels'], result['scores']):
        bar = "/" * int(score * 25)
        print(f"   {label:<15} {bar:<25} {score:.1%}")

Candidate labels: ['technology', 'politics', 'sports', 'business', 'science', 'entertainment']


 Text: The new transformer architecture reduces inference time by 40% on benchmark tasks.
   Top label : TECHNOLOGY (94.2%)  [5059ms]
   Runner-up : BUSINESS (2.0%)
   technology      ///////////////////////   94.2%
   business                                  2.0%
   entertainment                             1.3%
   science                                   1.2%
   sports                                    0.9%
   politics                                  0.4%

 Text: The president signed a new climate bill into law amid rising global temperatures.
   Top label : SCIENCE (43.8%)  [4250ms]
   Runner-up : POLITICS (36.9%)
   science         //////////                43.8%
   politics        /////////                 36.9%
   technology      /                         6.2%
   business        /                         5.6%
   sports                                    3.8%
   entertainment      

In [36]:
# ── Multi-label mode: a single text can match MULTIPLE labels ────────────────
# Important: multi_label=True changes the underlying NLI computation

crossover_text = "The NASA budget cut will affect satellite research and thousands of engineering jobs."

print(f"Text: {crossover_text}\n")
print("─── multi_label=False (forced single choice) ─────────\n")

r1 = zero_shot(crossover_text, candidate_labels=ZS_LABELS, multi_label=False)
for label, score in zip(r1['labels'][:3], r1['scores'][:3]):
    print(f"  {label:<15} {score:.1%}")

print("\n─── multi_label=True (each label scored independently) ─\n")

r2 = zero_shot(crossover_text, candidate_labels=ZS_LABELS, multi_label=True)
for label, score in zip(r2['labels'][:3], r2['scores'][:3]):
    print(f"  {label:<15} {score:.1%}")

print("\n > Multi-label is better for content tagging.")
print(" > Single-label is better for exclusive categorization.")

Text: The NASA budget cut will affect satellite research and thousands of engineering jobs.

─── multi_label=False (forced single choice) ─────────

  technology      58.3%
  science         36.3%
  business        1.9%

─── multi_label=True (each label scored independently) ─

  technology      95.8%
  science         88.9%
  politics        1.3%

 > Multi-label is better for content tagging.
 > Single-label is better for exclusive categorization.


In [37]:
# ── Power feature: completely custom label sets per request ──────────────────
# No re-training needed. This is what makes zero-shot so powerful.

email = "Hi, I tried to reset my password but the link expired. Can you help?"

for label_set_name, labels in [
    ("Support category",  ["billing", "technical issue", "account access", "feature request"]),
    ("Urgency",           ["urgent", "normal", "low priority"]),
    ("Tone",              ["frustrated", "neutral", "happy"]),
]:
    r = zero_shot(email, candidate_labels=labels, multi_label=False)
    top = r['labels'][0]
    score = r['scores'][0]
    print(f"  {label_set_name:<20} → {top} ({score:.1%})")

print("\n Same model, same email — 3 completely different classification tasks!")

  Support category     → technical issue (49.3%)
  Urgency              → urgent (88.0%)
  Tone                 → frustrated (97.5%)

 Same model, same email — 3 completely different classification tasks!


---
##  Pipeline 4 — Named Entity Recognition (NER)
**Model:** `dslim/bert-base-NER`  
**What it does:** Identifies and labels named entities — `PER` (person), `ORG` (organization), `LOC` (location), `MISC` (miscellaneous).

**Key concept — B-I-O tagging:**  
- `B-PER` = **B**eginning of a Person entity  
- `I-PER` = **I**nside a Person entity (continuation)  
- `O` = **O**utside any entity  

The model predicts a tag for *each token*. The pipeline aggregates them into spans for you.

In [38]:
print("Loading NER model...")
ner = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"  # Merges B-I-O tokens into full entity spans
)
print("NER model ready")

Loading NER model...


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cpu


NER model ready


In [43]:
# ── Run NER on all test texts ────────────────────────────────────────────────
ENTITY_LABELS = {'PER': '👤', 'ORG': '🏢', 'LOC': '📍', 'MISC': '🔵'}

for text in TEXTS:
    result, ms = timed_run(ner, text)
    print(f"\n {text}\n")

    if not result:
        print("  → No entities found")
        continue

    for ent in result:
        icon = ENTITY_LABELS.get(ent['entity_group'], '❓')
        print(f"   {icon} {ent['entity_group']:<6} '{ent['word']}'  (score: {ent['score']:.1%}, pos: {ent['start']}-{ent['end']})")


 I absolutely love how HuggingFace makes state-of-the-art NLP so accessible!

   🔵 MISC   'Hu'  (score: 87.8%, pos: 22-24)
   🏢 ORG    '##F'  (score: 91.2%, pos: 29-30)
   🏢 ORG    'NLP'  (score: 86.6%, pos: 57-60)

 The model was painfully slow and the documentation is completely outdated.

  → No entities found

 The library released version 4.0 last Tuesday.

  → No entities found

 Elon Musk announced that Tesla will open a new factory in Berlin next year.

   🏢 ORG    'Elon Musk'  (score: 99.8%, pos: 0-9)
   🏢 ORG    'Tesla'  (score: 98.8%, pos: 25-30)
   📍 LOC    'Berlin'  (score: 100.0%, pos: 58-64)

 Apple and Google are collaborating on a new AI safety initiative in California.

   🏢 ORG    'Apple'  (score: 99.8%, pos: 0-5)
   🏢 ORG    'Google'  (score: 99.8%, pos: 10-16)
   🔵 MISC   'AI'  (score: 99.8%, pos: 44-46)
   📍 LOC    'California'  (score: 100.0%, pos: 68-78)


In [46]:
# ── Entity highlighting ──────────
# Annotates the original text with [ENTITY/TYPE] markup

def highlight_entities(text, pipe):
    """Returns a string with entities wrapped in [word/TYPE] tags."""
    entities = pipe(text)

    if not entities:
        return text

    # Sort by start position, then replace from right-to-left to preserve indices
    entities_sorted = sorted(entities, key=lambda x: x['start'], reverse=True)
    annotated = text
    for ent in entities_sorted:
        start, end = ent['start'], ent['end']
        label = ent['entity_group']
        icon = ENTITY_LABELS.get(label, '?')
        annotated = annotated[:start] + f"[{text[start:end]}/{label}]" + annotated[end:]

    return annotated

print("Entity highlighting demo:\n")
for text in TEXTS[3:]:  # These two have the most entities
    highlighted = highlight_entities(text, ner)
    print(f"Input : {text}")
    print(f"Output: {highlighted}")
    print()

Entity highlighting demo:

Input : Elon Musk announced that Tesla will open a new factory in Berlin next year.
Output: [Elon Musk/ORG] announced that [Tesla/ORG] will open a new factory in [Berlin/LOC] next year.

Input : Apple and Google are collaborating on a new AI safety initiative in California.
Output: [Apple/ORG] and [Google/ORG] are collaborating on a new [AI/MISC] safety initiative in [California/LOC].



In [47]:
# ── Aggregation strategies — understanding the difference ───────────────────
# This matters when multi-token entity names are split differently

test_ner_text = "Sundar Pichai, CEO of Google, met with Ursula von der Leyen at the European Commission in Brussels."

for strategy in ["simple", "first", "average", "max"]:
    ner_strat = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy=strategy)
    result = ner_strat(test_ner_text)
    entities = [(e['word'], e['entity_group'], f"{e['score']:.0%}") for e in result]
    print(f"strategy='{strategy}':\n  {entities}\n")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another 

strategy='simple':
  [('Sundar Pichai', 'PER', '100%'), ('Google', 'ORG', '100%'), ('Ursula von der Leyen', 'PER', '93%'), ('European Commission', 'ORG', '100%'), ('Brussels', 'LOC', '100%')]



Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


strategy='first':
  [('Sundar Pichai', 'PER', '100%'), ('Google', 'ORG', '100%'), ('Ursula von der Leyen', 'PER', '100%'), ('European Commission', 'ORG', '100%'), ('Brussels', 'LOC', '100%')]



Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


strategy='average':
  [('Sundar Pichai', 'PER', '75%'), ('Google', 'ORG', '100%'), ('Ursula von der Leyen', 'PER', '96%'), ('European Commission', 'ORG', '100%'), ('Brussels', 'LOC', '100%')]

strategy='max':
  [('Sundar Pichai', 'PER', '100%'), ('Google', 'ORG', '100%'), ('Ursula von der Leyen', 'PER', '100%'), ('European Commission', 'ORG', '100%'), ('Brussels', 'LOC', '100%')]



---
## Pipeline 5 — Question Answering
**Model:** `deepset/roberta-base-squad2`  
**Type:** Extractive QA — the model identifies the *span* in the context that answers the question. It does not generate new text.

**How it works:** The model predicts a `start` and `end` token position within the context paragraph. The answer is the slice `context[start:end]`.

> **Important:** If the answer is not in the context, the model returns a low-score empty string. We handle this as an edge case below.

In [48]:
print("Loading QA model...")
qa = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2"
)
print("QA model ready")

Loading QA model...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

Device set to use cpu


QA model ready


In [53]:
# ── Answer all questions from the HuggingFace context ───────────────────────
print("Context:")
print(f"  {QA_CONTEXT[:120]}...\n")
print("=" * 60)

for question in QA_QUESTIONS:
    result, ms = timed_run(qa, question=question, context=QA_CONTEXT)
    bar = "█" * int(result['score'] * 20)
    print(f"\nQ: {question}")
    print(f"A: '{result['answer']}'")
    print(f"   Confidence: [{bar:<20}] {result['score']:.1%}  (pos: {result['start']}-{result['end']}, {ms}ms)")

Context:
  HuggingFace was founded in 2016 by Clement Delangue, Julien Chaumond, and Thomas Wolf.
The company is headquartered in N...


Q: Who founded HuggingFace?
A: 'Clement Delangue, Julien Chaumond, and Thomas Wolf'
   Confidence: [██████████████████  ] 94.1%  (pos: 35-85, 479ms)

Q: Where is HuggingFace headquartered?
A: 'New York City and Paris'
   Confidence: [█████████████████   ] 85.9%  (pos: 119-142, 467ms)

Q: How many models does the Hub host?
A: 'over 500,000'
   Confidence: [██████████          ] 50.4%  (pos: 294-306, 489ms)

Q: What was HuggingFace valued at in 2023?
A: '$4.5 billion'
   Confidence: [█                   ] 7.0%  (pos: 412-424, 486ms)


In [59]:
# ── Edge case: question not answerable from the context ─────────────────────
# The model returns something, but with very low confidence — we must detect this!

UNANSWERABLE_THRESHOLD = 0.10  # Below this, mark as unanswerable

unanswerable_questions = [
    "Who is the current president of the United States?",  # Not in context
    "What is HuggingFace's stock ticker symbol?",          # Not mentioned
    "When was the Transformers library first released?",   # Not in context
]

print("Unanswerable question handling:\n")
for q in unanswerable_questions:
    result = qa(question=q, context=QA_CONTEXT)
    if result['score'] < UNANSWERABLE_THRESHOLD or result['answer'].strip() == '':
        print(f"Q: {q}")
        print(f"   ⚠️  Cannot answer from context. (model returned: {result['answer']}  {result['score']:.1%})\n")
    else:
        print(f"Q: {q}")
        print(f"   ✅ '{result['answer']}' ({result['score']:.1%})\n")



Unanswerable question handling:

Q: Who is the current president of the United States?
   ⚠️  Cannot answer from context. (model returned: 
vision, and audio tasks.  0.0%)

Q: What is HuggingFace's stock ticker symbol?
   ⚠️  Cannot answer from context. (model returned: 
Transformers  0.1%)

Q: When was the Transformers library first released?
   ⚠️  Cannot answer from context. (model returned: 2016  0.0%)



In [60]:
# ── Top-k answers: get multiple candidate answers with confidence ─────────────
# Useful for showing alternatives in the UI

question = "Who founded HuggingFace?"
result = qa(question=question, context=QA_CONTEXT, top_k=3)

print(f"Q: {question}")
print(f"Context snippet: {QA_CONTEXT[:100]}...\n")
print("Top-3 candidate answers:")
for i, r in enumerate(result, 1):
    print(f"  {i}. '{r['answer']}'  →  score: {r['score']:.1%}  (pos {r['start']}-{r['end']})")

Q: Who founded HuggingFace?
Context snippet: HuggingFace was founded in 2016 by Clement Delangue, Julien Chaumond, and Thomas Wolf.
The company i...

Top-3 candidate answers:
  1. 'Clement Delangue, Julien Chaumond, and Thomas Wolf'  →  score: 94.1%  (pos 35-85)
  2. 'Clement Delangue, Julien Chaumond, and Thomas Wolf.'  →  score: 0.4%  (pos 35-86)
  3. 'Clement Delangue, Julien Chaumond'  →  score: 0.1%  (pos 35-68)


---


In [62]:
def run_all_pipelines(text, context=None, qa_question=None, zs_labels=None):
    """
    Runs all 5 NLP pipelines on the input text.
    Returns a unified dict — this will become our FastAPI response schema.

    Parameters
    ----------
    text         : The main text to analyze.
    context      : Context paragraph for QA (optional, uses text if not given).
    qa_question  : Question for QA pipeline (optional).
    zs_labels    : List of candidate labels for zero-shot (optional).
    """
    results = {"input_text": text[:80] + "..." if len(text) > 80 else text}

    # 1. Summarization
    if len(text.split()) > 30:
        r, ms = timed_run(summarizer, text, max_length=60, min_length=15, do_sample=False)
        results["summarization"] = {"summary": r[0]["summary_text"], "latency_ms": ms}
    else:
        results["summarization"] = {"summary": None, "note": "Text too short to summarize"}

    # 2. Sentiment
    r, ms = timed_run(sentiment_primary, text)
    top = max(r[0], key=lambda x: x['score'])
    results["sentiment"] = {
        "label": top['label'],
        "confidence": round(top['score'], 4),
        "low_confidence": top['score'] < CONFIDENCE_THRESHOLD,
        "latency_ms": ms
    }

    # 3. Zero-shot
    labels = zs_labels or ["technology", "politics", "sports", "business", "science"]
    r, ms = timed_run(zero_shot, text, candidate_labels=labels, multi_label=False)
    results["zero_shot"] = {
        "top_label": r['labels'][0],
        "top_score": round(r['scores'][0], 4),
        "all": [{'label': l, 'score': round(s, 4)} for l, s in zip(r['labels'], r['scores'])],
        "latency_ms": ms
    }

    # 4. NER
    r, ms = timed_run(ner, text)
    results["ner"] = {
        "entities": [{'word': e['word'], 'type': e['entity_group'], 'score': round(e['score'], 4)} for e in r],
        "count": len(r),
        "latency_ms": ms
    }

    # 5. QA
    if qa_question:
        ctx = context or text
        r, ms = timed_run(qa, question=qa_question, context=ctx)
        results["question_answering"] = {
            "question": qa_question,
            "answer": r['answer'] if r['score'] >= UNANSWERABLE_THRESHOLD else None,
            "confidence": round(r['score'], 4),
            "answerable": r['score'] >= UNANSWERABLE_THRESHOLD,
            "latency_ms": ms
        }

    return results

print("Batch function defined")

Batch function defined


In [65]:
# ── Run the batch function on a news-style text ──────────────────────────────
sample = """Sam Altman, CEO of OpenAI, announced a new partnership with Microsoft worth
over $10 billion to accelerate AGI research in Seattle. The announcement was met
with excitement from the AI community but raised concerns among regulators in Brussels."""

output = run_all_pipelines(
    text=sample,
    qa_question="How much is the partnership worth?"
)

print(json.dumps(output, indent=2, default=float))

total_ms = sum(v.get('latency_ms', 0) for v in output.values() if isinstance(v, dict))
print(f"\n⏱️  Total inference time: {total_ms}ms")

Your max_length is set to 60, but your input_length is only 52. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=26)


{
  "input_text": "Sam Altman, CEO of OpenAI, announced a new partnership with Microsoft worth \nove...",
  "summarization": {
    "summary": "Sam Altman, CEO of OpenAI, announced a new partnership with Microsoft worth over $10 billion to accelerate AGI research in Seattle. The announcement was met with excitement from the AI community but raised concerns among regulators in Brussels.",
    "latency_ms": 12972
  },
  "sentiment": {
    "label": "positive",
    "confidence": 0.5137,
    "low_confidence": true,
    "latency_ms": 250
  },
  "zero_shot": {
    "top_label": "technology",
    "top_score": 0.6639,
    "all": [
      {
        "label": "technology",
        "score": 0.6639
      },
      {
        "label": "business",
        "score": 0.1719
      },
      {
        "label": "science",
        "score": 0.124
      },
      {
        "label": "politics",
        "score": 0.0269
      },
      {
        "label": "sports",
        "score": 0.0133
      }
    ],
    "latency_ms": 